In [1]:
import os
import sys
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)
# Set the parent directory as the current directory
os.chdir(parent_dir)

In [ ]:
from rdma.utils.llm_client import LocalLLMClient

llm_client = LocalLLMClient(model_type="mistral_24b", device="cuda:0",temperature=0.0001)


Initialized ModelLoader with cache directory: /shared/rsaas/jw3/rare_disease/model_cache
Loading LLM!
Device configuration: cuda:0
Using device map: {'': 'cuda:0'}
Loading 70B model with quantization: mistral_24b
Generated cache path: /shared/rsaas/jw3/rare_disease/model_cache/Mistral-Small-24B-Instruct-2501_4bit_nf4
Valid cache found at /shared/rsaas/jw3/rare_disease/model_cache/Mistral-Small-24B-Instruct-2501_4bit_nf4
Loading cached quantized model from /shared/rsaas/jw3/rare_disease/model_cache/Mistral-Small-24B-Instruct-2501_4bit_nf4


/home/johnwu3/miniconda3/envs/hporag/lib/python3.10/site-packages/transformers/quantizers/auto.py:206: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Device set to use cuda:0


Hello! I'm here to help. How can I assist you today? If you have any medical questions or need information on a specific topic, feel free to ask. Please note that while I strive to provide accurate and helpful information, I am an AI and my knowledge cutoff is 2023, I don't have real-time web browsing capabilities, and I can't provide medical advice. For medical advice, please consult a healthcare professional. Here are a few examples of how I can help:

* Explain medical terms or concepts
* Provide information on diseases, conditions, or treatments
* Offer insights into medical procedures or tests
* Discuss healthcare trends or research (up to my knowledge cutoff in 2023)

What would you like to know or discuss?


# Benchmarking + More Robust LLM Evaluation in 1 Script.  Hit @ 1 (can be expanded to Hit @ 10) for later.

In [17]:
import json
import re
from typing import List, Dict, Tuple, Any

def parse_diseases_list(response: str) -> List[str]:
    """
    Robust function to parse LLM response into Python list
    """
    try:
        # First, try direct JSON parsing
        cleaned = response.strip()
        diseases_list = json.loads(cleaned)
        
        # Validate it's a list
        if isinstance(diseases_list, list):
            return diseases_list
        else:
            return []
            
    except json.JSONDecodeError:
        # Try to extract JSON array using regex
        json_pattern = r'\[(.*?)\]'
        match = re.search(json_pattern, response, re.DOTALL)
        
        if match:
            try:
                json_content = '[' + match.group(1) + ']'
                diseases_list = json.loads(json_content)
                return diseases_list
            except json.JSONDecodeError:
                pass
        
        # Last resort: manual parsing for common formats
        try:
            # Remove brackets and split by comma
            content = response.strip()
            content = re.sub(r'^\[|\]$', '', content)  # Remove outer brackets
            
            # Split by comma and clean each item
            items = [item.strip().strip('"\'') for item in content.split(',')]
            items = [item for item in items if item]  # Remove empty items
            
            return items
            
        except Exception:
            return []

def normalize_disease_name(disease: str) -> str:
    """
    Normalize disease name for comparison: lowercase and strip whitespace
    """
    return disease.lower().strip()

def benchmark_rare_disease_diagnosis(
    data: Dict[str, Any], 
    llm_client: Any, 
    num_samples: int = None,
    verbose: bool = False
) -> Dict[str, float]:
    """
    Benchmark rare disease diagnosis performance
    
    Args:
        data: Dictionary containing patient data with phenotypes and disease entities
        llm_client: LLM client with query method
        num_samples: Number of samples to evaluate (None for all)
        verbose: Whether to print detailed results for each case
    
    Returns:
        Dictionary with benchmark metrics
    """
    
    # System prompt for LLM
    diff_diag_sys_prompt = """Given the following phenotypes, identify the top 10 most likely rare diseases.

CRITICAL: Your response must be EXACTLY in this JSON format with no additional text:

["Disease Name 1", "Disease Name 2", "Disease Name 3", "Disease Name 4", "Disease Name 5", "Disease Name 6", "Disease Name 7", "Disease Name 8", "Disease Name 9", "Disease Name 10"]

Rules:
1. Return ONLY the JSON array - no explanations, no additional text
2. Use double quotes around disease names
3. Separate diseases with commas
4. First disease = most likely, last disease = least likely
5. If you find fewer than 10 diseases, that's okay
6. If you cannot find any rare diseases, return: []

Example of correct format:
["Marfan Syndrome", "Ehlers-Danlos Syndrome", "Osteogenesis Imperfecta"]

Your response:"""
    
    # Initialize counters
    total_diseases = 0
    hits = 0
    hits_at_1 = 0
    total_patients = 0
    patients_with_hits = 0
    parsing_failures = 0
    
    # Get samples to process
    items_to_process = list(data.items())
    if num_samples is not None:
        items_to_process = items_to_process[:num_samples]
    
    if verbose:
        print(f"Evaluating {len(items_to_process)} patients...")
        print("=" * 60)
    
    for patient_id, patient_data in items_to_process:
        if 'matched_phenotypes' not in patient_data:
            continue
            
        total_patients += 1
        patient_hits = 0
        
        # Build phenotypes string
        phenotypes = ""
        for phenotype in patient_data['matched_phenotypes']:
            phenotypes += phenotype['phenotype'] + ", "
        
        # Query LLM
        phenotypes_prompt = f"Phenotypes: {phenotypes}"
        llm_response = llm_client.query(
            system_message=diff_diag_sys_prompt, 
            user_input=phenotypes_prompt
        )
        
        # Parse response
        predicted_diseases = parse_diseases_list(llm_response)
        
        if not predicted_diseases:
            parsing_failures += 1
            if verbose:
                print(f"Patient {patient_id}: Failed to parse LLM response")
                print(f"Raw response: '{llm_response}'")
            continue
        
        # Normalize predicted diseases for comparison
        predicted_normalized = [normalize_disease_name(d) for d in predicted_diseases]
        
        # Get ground truth diseases
        observed_diseases = patient_data.get('disease_entities', [])
        
        if verbose:
            print(f"Patient ID: {patient_id}")
            print(f"Phenotypes: {phenotypes.strip(', ')}")
            print(f"Predicted diseases: {predicted_diseases}")
            print(f"Observed diseases: {observed_diseases}")
        
        # Calculate hits for this patient
        for observed_disease in observed_diseases:
            observed_normalized = normalize_disease_name(observed_disease)
            total_diseases += 1
            
            # Check if disease is in top-10 predictions
            hit_in_top10 = observed_normalized in predicted_normalized
            if hit_in_top10:
                hits += 1
                patient_hits += 1
                
                if verbose:
                    print(f"  ✓ Hit: '{observed_disease}' found in predictions")
            else:
                if verbose:
                    print(f"  ✗ Miss: '{observed_disease}' not found in predictions")
            
            # Check if disease is the top prediction (Hit@1)
            hit_at_1 = (observed_normalized == predicted_normalized[0] 
                       if predicted_normalized else False)
            if hit_at_1:
                hits_at_1 += 1
                if verbose:
                    print(f"  ✓ Hit@1: '{observed_disease}' is top prediction")
        
        if patient_hits > 0:
            patients_with_hits += 1
        
        if verbose:
            print(f"  Patient hits: {patient_hits}/{len(observed_diseases)}")
            print("-" * 60)
    
    # Calculate final metrics
    hit_rate = hits / total_diseases if total_diseases > 0 else 0
    hit_at_1_rate = hits_at_1 / total_diseases if total_diseases > 0 else 0
    patient_hit_rate = patients_with_hits / total_patients if total_patients > 0 else 0
    parsing_success_rate = 1 - (parsing_failures / total_patients) if total_patients > 0 else 0
    
    results = {
        'hit_rate': hit_rate,
        'hit_at_1_rate': hit_at_1_rate,
        'patient_hit_rate': patient_hit_rate,
        'parsing_success_rate': parsing_success_rate,
        'total_diseases': total_diseases,
        'total_patients': total_patients,
        'hits': hits,
        'hits_at_1': hits_at_1,
        'patients_with_hits': patients_with_hits,
        'parsing_failures': parsing_failures
    }
    
    return results

def print_benchmark_results(results: Dict[str, float]) -> None:
    """
    Print benchmark results in a formatted way
    """
    print("\n" + "=" * 50)
    print("RARE DISEASE DIAGNOSIS BENCHMARK RESULTS")
    print("=" * 50)
    print(f"Total patients evaluated: {results['total_patients']}")
    print(f"Total diseases to predict: {results['total_diseases']}")
    print(f"LLM parsing success rate: {results['parsing_success_rate']:.2%}")
    print("-" * 50)
    print(f"Hit Rate (Top-10): {results['hit_rate']:.2%} ({results['hits']}/{results['total_diseases']})")
    print(f"Hit@1 Rate: {results['hit_at_1_rate']:.2%} ({results['hits_at_1']}/{results['total_diseases']})")
    print(f"Patient Hit Rate: {results['patient_hit_rate']:.2%} ({results['patients_with_hits']}/{results['total_patients']})")
    print("=" * 50)

# Example usage:
if __name__ == "__main__":
    # Example of how to use the function
    from rdma.utils.data import read_json_file
    
    # Load data
    data = read_json_file("data/medical_students_data/high_agreement_with_phenotypes.json")
    
    # Run benchmark on first 5 patients with verbose output
    results = benchmark_rare_disease_diagnosis(
        data=data, 
        llm_client=llm_client,  # Your LLM client
        num_samples=2,
        verbose=True
    )
    
    # Print results
    print_benchmark_results(results)

Evaluating 2 patients...
Patient ID: 10402135
Phenotypes: shortness of breath, history of COPD, dementia, HTN, Afib, gerd, polio, left ankle paralysis, dyspnea, increasing dyspnea, cough, wheezing, mild epigastric pain, confused from her baseline, hallucinations, swollen right leg, tender to palpation, elevated systolic blood pressure, elevated reticulocyte count, hypoxia, leukocytosis, hyponatremia, decreased lactate, b/l effusions, possible consolidation, elevated A/G Ratio, sweats, nausea, vomiting, diarrhea, constipation, dark tarry, frequency, hematuria, Former alcohol abuse, Delusional disorder, auditory hallucination, raynaud's phenomenon, Diverticulosis, Colonic adenoma, left bundle branch block, hypertension, anxiety, cognitive decline, urinary incontinence, urge, Hearing difficulty, depression, breast cancer, frail appearing, hard of hearing, scattered rhonchi, rales, rales, decreased breath sounds at bases, systolic murmur, 1+ edema ___ to knee, 2+ edema, tongue protrudes sy